# Kapitel 9 — Die Zahl der Dimensionen verringern · Notebook zum Ausfüllen

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch09_a_completer.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Die zu bearbeitenden Zellen sind mit **ZU ERGÄNZEN** gekennzeichnet. Führen Sie die Zellen der Reihe nach aus: Solange eine Antwort fehlt, melden die Prüfungen dies ohne Fehler. Die Lösung finden Sie in `ch09_corrige.ipynb` auf derselben Begleitseite.

**Konvention.** Wie im Buch teilen Mittelwerte, Varianzen und Kovarianzen durch $n$.

**Inhalt.** Daten · Die Rechnungen der Lektionen 9.1 bis 9.3 · Aufgaben 9.1 bis 9.6

**Bevor Sie Jupyter öffnen:** Die Aufgaben dieses Kapitels werden zuerst auf Papier gelöst, mit einem Taschenrechner. Die Zellen unten dienen dazu, Ihre Ergebnisse zu **überprüfen**.

**Farbcode:** Blau = Wohnungen; schwarze Raute = G; Orange = Projektionen; Magenta gestrichelt = was die Projektion verliert.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farbcode des Buchs: Wohnungen in Blau, G als schwarze Raute, Projektionen in Orange,
# was die Projektion verliert, in Magenta (gestrichelt)
BLUE, ORANGE, MAGENTA, INK = '#2a78d6', '#eb6834', '#c2185b', '#0b0b0b'


def project(X, u):
    # Koordinaten der Punkte (Zeilen von X) auf der Richtung u, auf Norm 1 gebracht (Gleichung 9.1)
    u = np.asarray(u, dtype=float)
    return np.asarray(X, dtype=float) @ (u / np.linalg.norm(u))


def cov_matrix(X):
    # Kovarianzmatrix der Spalten von X, mit Division durch n (Gleichung 9.2)
    X = np.asarray(X, dtype=float)
    D = X - X.mean(axis=0)
    return D.T @ D / len(X)

## Daten

Die Kennung D2.1 ist die des Buchs und der CSV-Datei. Die kleinen Datensätze der Aufgaben sind aus den Aufgabenstellungen übernommen.

In [ ]:
# Daten von Kapitel 9, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D2.1 — die sieben Wohnungen (A bis F: Referenzverkäufe; G: die zu schätzende Wohnung)
# Spalten der CSV-Datei → Namen im Notebook: logement → home, surface_m2 → floor_area (m²),
# distance_km → distance (km, Entfernung zum Zentrum), prix_kEUR → price (k€)
homes = pd.DataFrame([
    ('A', 32, 3.0, 118),
    ('B', 45, 1.5, 172),
    ('C', 54, 2.5, 185),
    ('D', 70, 1.0, 245),
    ('E', 85, 2.5, 279),
    ('F', 98, 0.5, 335),
    ('G', 62, 2.5, 209)
], columns=['home', 'floor_area', 'distance', 'price'])

reference = homes[homes['home'] != 'G']          # A bis F: Referenz-Mittelwerte und -Standardabweichungen
mu = reference[['floor_area', 'distance']].mean().to_numpy()
sigma = reference[['floor_area', 'distance']].std(ddof=0).to_numpy()   # Division durch n
Z = (reference[['floor_area', 'distance']].to_numpy() - mu) / sigma   # standardisierte Werte von A bis F
z_G = (homes.loc[homes['home'] == 'G', ['floor_area', 'distance']].to_numpy()[0] - mu) / sigma
names = list(reference['home'])
print(homes.to_string(index=False))

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `home` | Kennung der Wohnung (A, B…) | — | `logement` |
| `floor_area` | Wohnfläche der Wohnung | m² | `surface_m2` |
| `price` | Verkaufspreis | k€ | `prix_kEUR` |
| `distance` | Entfernung zum Stadtzentrum | km | `distance_km` |
| `trainee` | Kennung des Praktikanten | — | — |
| `math_score` | Mathematiknote | Punkte von 20 | — |
| `writing_score` | Aufsatznote | Punkte von 20 | — |

## Die Rechnungen der Lektionen, in Python

Diese Zellen wiederholen die Rechnungen der Lektionen 9.1 bis 9.3.

In [ ]:
# Lektion 9.1 — auf eine Achse projizieren (Gleichung 9.1): Achse der Fläche, dann Diagonale u = (1; −1)/√2
u_area = np.array([1.0, 0.0])
u_diag = np.array([1.0, -1.0]) / np.sqrt(2)
t = project(Z, u_diag)
t_G = project(z_G, u_diag)
print(pd.DataFrame({'home': names, 'z_area': Z[:, 0], 'z_distance': Z[:, 1], 't': t}).round(3).to_string(index=False))
print(f"G: t = {t_G:.3f}")
print(f"erhaltene Varianz: Achse der Fläche {np.mean(project(Z, u_area) ** 2):.3f}, Diagonale {np.mean(t ** 2):.3f}")
print(f"Gesamtvarianz {np.sum(Z.var(axis=0)):.3f} = erhalten {np.mean(t ** 2):.3f} + verloren {np.sum(Z.var(axis=0)) - np.mean(t ** 2):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, u, title in zip(axes, (u_area, u_diag), ('Achse der Fläche', 'Diagonale')):
    shadows = np.outer(project(Z, u), u)
    for p, s in zip(Z, shadows):
        ax.plot([p[0], s[0]], [p[1], s[1]], color=MAGENTA, linestyle='--', linewidth=1)
    ax.axline((0, 0), u, color=ORANGE, linewidth=1)
    ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, zorder=3)
    ax.scatter(shadows[:, 0], shadows[:, 1], color=ORANGE, zorder=3)
    ax.scatter([z_G[0]], [z_G[1]], marker='D', color=INK, zorder=3)
    for n, p in zip(names + ['G'], list(Z) + [z_G]):
        ax.annotate(n, p, xytext=(4, 4), textcoords='offset points')
    ax.set_aspect('equal'); ax.set_title(title); ax.set_xlabel('z Fläche'); ax.set_ylabel('z Entfernung')
plt.tight_layout(); plt.show()

In [ ]:
# Lektion 9.2 — die Kovarianzmatrix (Gleichung 9.2) und ihre Eigenrichtungen
products = Z[:, 0] * Z[:, 1]
print('Produkte z_s × z_d:', products.round(3), '→ Summe', round(products.sum(), 3))
C = cov_matrix(Z)
print('Kovarianzmatrix C =\n', C.round(3))
for vec in ([1, 0], [1, -1], [1, 1]):
    print(f"C × {vec} = {(C @ np.array(vec, dtype=float)).round(3)}")
u1, u2 = u_diag
print(f"Varianz auf der Diagonale, allein mit C: {C[0, 0] * u1 ** 2 + 2 * C[0, 1] * u1 * u2 + C[1, 1] * u2 ** 2:.3f}")

In [ ]:
# Lektion 9.3 — die PCA: Eigenwerte in absteigender Reihenfolge, Anteil der erhaltenen Varianz (Gleichung 9.3)
eigenvalues, eigenvectors = np.linalg.eigh(C)          # eigh: symmetrische Matrix, aufsteigende Reihenfolge
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
v1 = eigenvectors[:, 0] * np.sign(eigenvectors[0, 0])  # eine Richtung und ihre Gegenrichtung beschreiben dieselbe Achse
print('Eigenwerte:', eigenvalues.round(3), '→ Summe', round(eigenvalues.sum(), 3))
print('erste Eigenrichtung:', v1.round(3))
print('erhaltene Anteile (%):', (100 * eigenvalues / eigenvalues.sum()).round(1))
pc1 = project(Z, v1)
print('Verkäufe nach erster Komponente geordnet:', [names[i] for i in np.argsort(pc1)],
      '; Preise (k€):', list(reference['price'].to_numpy()[np.argsort(pc1)]))

# B und E: nah auf der Komponente, weit voneinander entfernt in der Ebene
q = project(Z, [1, 1])
iB, iE = names.index('B'), names.index('E')
print(f"B und E: Differenz auf der Komponente {abs(pc1[iE] - pc1[iB]):.2f}, Abstand in der Ebene "
      f"{np.linalg.norm(Z[iE] - Z[iB]):.2f}, Differenz auf der zweiten Komponente {abs(q[iE] - q[iB]):.2f}")

# Zur Vertiefung: Die SVD von Z liefert dieselben Eigenwerte, s_k² / n
s = np.linalg.svd(Z, compute_uv=False)
print(f"Singulärwerte: {s.round(3)}; s² / n = {(s ** 2 / len(Z)).round(3)}")

## Aufgabe 9.1 — Auf zwei Achsen projizieren

*Begriffe: Projektion und erhaltene Varianz (Lektion 9.1, Gleichung 9.1).* Drei Punkte, $P_1 = (4 ; 2)$, $P_2 = (1 ; 3)$ und $P_3 = (-2 ; 1)$, und zwei Richtungen, $\mathbf{u} = (0{,}8 ; 0{,}6)$ und $\mathbf{v} = (-0{,}6 ; 0{,}8)$.

### a) bis d) Auf Papier

*Ihre Rechnungen:* …

In [ ]:
# Die drei Punkte (eine Zeile pro Punkt) und die zwei Richtungen
points = np.array([[4, 2], [1, 3], [-2, 1]], dtype=float)
u = np.array([0.8, 0.6])
v = np.array([-0.6, 0.8])

In [ ]:
# ZU ERGÄNZEN: Koordinaten auf u und auf v, dann ihre Varianzen (Division durch 3)
coords_u = None   # ZU ERGÄNZEN: points @ u
coords_v = None   # ZU ERGÄNZEN
var_u = None      # ZU ERGÄNZEN: np.var(coords_u) teilt durch n
var_v = None      # ZU ERGÄNZEN

In [ ]:
# Prüfung von a) bis d)
if coords_u is None or coords_v is None or var_u is None or var_v is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die vier Zeilen.')
else:
    print(f"a) Normen {np.linalg.norm(u):.3f} und {np.linalg.norm(v):.3f}, Skalarprodukt {u @ v:.3f}")
    print('b) auf u:', np.round(coords_u, 2), '; auf v:', np.round(coords_v, 2), '→',
          'richtig' if np.allclose(coords_u, [4.4, 2.6, -1.0]) and np.allclose(coords_v, [-0.8, 1.8, 2.0]) else 'bitte prüfen')
    print(f"c) P1: {coords_u[0] ** 2 + coords_v[0] ** 2:.2f} gegenüber einer quadrierten Norm von {points[0] @ points[0]:.2f}")
    total = np.sum(points.var(axis=0))
    print(f"d) Varianzen {var_u:.2f} und {var_v:.2f}, Summe {var_u + var_v:.2f}, Gesamtvarianz {total:.2f} →",
          'richtig' if abs(var_u - 5.04) < 1e-9 and abs(var_v - 1.6267) < 1e-3 and abs(var_u + var_v - total) < 1e-9 else 'bitte prüfen')

## Aufgabe 9.2 — Eine Kovarianzmatrix von Hand

*Begriffe: Kovarianz (Lektion 1.5), Kovarianzmatrix (Lektion 9.2, Gleichung 9.2).* Vier Praktikanten haben zwei Prüfungen abgelegt, bewertet mit Punkten von 20 (erfundener Datensatz).

- `trainee` — Praktikant, S1 bis S4;
- `math_score` — Note in der Rechenprüfung, von 20, in Punkten;
- `writing_score` — Note in der Aufsatzprüfung, von 20, in Punkten.

In [ ]:
# Die vier Praktikanten (Aufgaben 9.2 bis 9.5)
trainees = pd.DataFrame({
    'trainee':       ['S1', 'S2', 'S3', 'S4'],
    'math_score':    [15, 13, 9, 11],
    'writing_score': [11, 13, 9, 7],
})
scores = trainees[['math_score', 'writing_score']].to_numpy(dtype=float)
print(trainees.to_string(index=False))

### a) bis c) Auf Papier

*Ihre Rechnungen:* …

In [ ]:
# ZU ERGÄNZEN: Abweichungen vom Mittelwert, Kovarianzmatrix (Division durch 4) und Korrelation r
deviations = None   # ZU ERGÄNZEN: scores - scores.mean(axis=0)
M = None            # ZU ERGÄNZEN: deviations.T @ deviations / 4
r = None            # ZU ERGÄNZEN: M[0, 1] / Wurzel aus M[0, 0] × M[1, 1]

In [ ]:
# Prüfung von a) bis c)
if deviations is None or M is None or r is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die drei Zeilen.')
else:
    print('a) Mittelwerte:', scores.mean(axis=0), '; Abweichungen:', deviations.tolist())
    print('b) und c) M =', np.asarray(M).tolist(), f"; r = {r:.2f} →",
          'richtig' if np.allclose(M, [[5, 3], [3, 5]]) and abs(r - 0.6) < 1e-9 else 'bitte prüfen')
    print('Kontrolle mit cov_matrix:', cov_matrix(scores).tolist())

## Aufgabe 9.3 — Die Eigenrichtungen finden

*Begriffe: Eigenrichtung und Eigenwert (Lektion 9.2).* Wir nehmen die Matrix aus Aufgabe 9.2 wieder auf, $M = \begin{pmatrix} 5 & 3 \\ 3 & 5 \end{pmatrix}$.

### a) bis d) Auf Papier

*Ihre Rechnungen:* …

In [ ]:
# ZU ERGÄNZEN: Eigenwerte von M in absteigender Reihenfolge und der von der besten Richtung erhaltene Anteil
M93 = np.array([[5.0, 3.0], [3.0, 5.0]])
eigenvalues_93 = None   # ZU ERGÄNZEN: np.linalg.eigh(M93) gibt (Werte, Vektoren) in aufsteigender Reihenfolge zurück
best_share = None       # ZU ERGÄNZEN: größter Eigenwert / Summe der Eigenwerte

In [ ]:
# Prüfung von a) bis d)
if eigenvalues_93 is None or best_share is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    for vec in ([1, 0], [1, 1], [1, -1]):
        print(f"a) und b) M × {vec} = {(M93 @ np.array(vec, dtype=float)).tolist()}")
    print(f"Eigenwerte {eigenvalues_93.round(3).tolist()}, Anteil der besten Richtung {100 * best_share:.0f} % →",
          'richtig' if np.allclose(eigenvalues_93, [8, 2]) and abs(best_share - 0.8) < 1e-9 else 'bitte prüfen')
    r_houses = -0.603
    print('d) r = −0,603: Eigenwerte', np.sort(np.linalg.eigh(np.array([[1, r_houses], [r_houses, 1]]))[0])[::-1].round(3).tolist())

## Aufgabe 9.4 — Eine erste Komponente berechnen

*Begriffe: PCA (Lektion 9.3), Projektion (Gleichung 9.1).* Wir nehmen die Abweichungen vom Mittelwert aus Aufgabe 9.2 wieder auf. Hier wird nicht standardisiert: Die beiden Prüfungen haben dieselbe Einheit und dieselbe Varianz. Die erste Eigenrichtung ist $\mathbf{v}_1 = (1 ; 1)/\sqrt{2}$.

### a) bis d) Auf Papier

*Ihre Rechnungen und Ihre Interpretation:* …

In [ ]:
# ZU ERGÄNZEN: Koordinaten auf v1, Varianz, dann Rekonstruktion der Noten von S1
v1_trainees = np.array([1.0, 1.0]) / np.sqrt(2)
t_trainees = None        # ZU ERGÄNZEN: Abweichungen vom Mittelwert (Aufgabe 9.2) @ v1_trainees
reconstructed_S1 = None  # ZU ERGÄNZEN: Mittelwerte + t_trainees[0] × v1_trainees

In [ ]:
# Prüfung von a) bis c)
if t_trainees is None or reconstructed_S1 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    var_t = np.mean(t_trainees ** 2)
    error = scores[0] - reconstructed_S1
    print('a) Koordinaten:', t_trainees.round(2).tolist(), f"; Varianz {var_t:.2f}")
    print(f"b) erhaltener Anteil {100 * var_t / np.sum(scores.var(axis=0)):.0f} %")
    print('c) rekonstruierte Noten von S1:', reconstructed_S1.round(2).tolist(), '; Fehler:', error.round(2).tolist(),
          f"; Länge {np.linalg.norm(error):.2f} →",
          'richtig' if np.allclose(t_trainees, [2 * np.sqrt(2)] * 2 + [-2 * np.sqrt(2)] * 2)
          and np.allclose(reconstructed_S1, [14, 12]) else 'bitte prüfen')

## Aufgabe 9.5 — Nah nach der Projektion?

*Begriffe: Grenzen der Projektion (Lektion 9.3, Gleichung 9.3).* Wir nehmen die Koordinaten aus Aufgabe 9.4 wieder auf.

### a) bis d) Auf Papier

*Ihre Rechnungen und Ihr Beweis:* …

In [ ]:
# ZU ERGÄNZEN: Abstände in der Ebene der beiden Noten
distance_S1_S2 = None   # ZU ERGÄNZEN: np.linalg.norm(scores[0] - scores[1])
distance_S1_S3 = None   # ZU ERGÄNZEN

In [ ]:
# Prüfung von a), c) und d)
if distance_S1_S2 is None or distance_S1_S3 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    t9 = (scores - scores.mean(axis=0)) @ (np.array([1.0, 1.0]) / np.sqrt(2))
    print(f"a) S1 und S2: projizierte Differenz {abs(t9[0] - t9[1]):.2f}, Abstand {distance_S1_S2:.2f}")
    print(f"c) S1 und S3: projizierte Differenz {abs(t9[0] - t9[2]):.2f} ≤ Abstand {distance_S1_S3:.2f} →",
          'richtig' if abs(distance_S1_S2 - np.sqrt(8)) < 1e-9 and abs(distance_S1_S3 - np.sqrt(40)) < 1e-9 else 'bitte prüfen')
    iB, iE = names.index('B'), names.index('E')
    t_BE = project(Z[[iB, iE]], [1, -1])
    q_BE = project(Z[[iB, iE]], [1, 1])
    print(f"d) B und E: Differenz auf der Komponente {abs(t_BE[1] - t_BE[0]):.2f}, Abstand {np.linalg.norm(Z[iE] - Z[iB]):.2f}, "
          f"verlorener Teil {abs(q_BE[1] - q_BE[0]):.2f}")

## Aufgabe 9.6 — Herausforderung: die PCA ohne Standardisierung

*Begriffe: Kovarianzmatrix (Gleichung 9.2), PCA und Einheiten (Lektion 9.3).* Für die Verkäufe A bis F arbeiten wir mit den rohen Abweichungen vom Mittelwert, ohne zu standardisieren.

- `floor_area` — Fläche der Wohnung, in m² (Datensatz D2.1, Spalte `surface_m2`);
- `distance` — Entfernung zum Zentrum, in km (Datensatz D2.1, Spalte `distance_km`).

### a) bis d) Auf Papier

*Ihre Rechnungen und Ihr Fazit:* …

In [ ]:
# ZU ERGÄNZEN: rohe Kovarianzmatrix von A bis F (Fläche in m², Entfernung in km), dann M × w
raw = reference[['floor_area', 'distance']].to_numpy(dtype=float)
M_raw = None   # ZU ERGÄNZEN: cov_matrix(raw)
w = np.array([1.0, -0.0238])
Mw = None      # ZU ERGÄNZEN: M_raw @ w

In [ ]:
# Prüfung von a) bis d)
if M_raw is None or Mw is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die beiden Zeilen.')
else:
    share = np.linalg.eigvalsh(M_raw)[-1] / np.trace(M_raw)
    raw_m = raw * [1, 1000]                       # Entfernung in Metern
    share_m = np.linalg.eigvalsh(cov_matrix(raw_m))[-1] / np.trace(cov_matrix(raw_m))
    print('a) und b) M =', M_raw.round(3).tolist(), '; M × w =', Mw.round(2).tolist(), f"; Verhältnis {Mw[1] / Mw[0]:.4f}")
    print(f"c) erhaltener Anteil {100 * share:.1f} % (Gesamtvarianz {np.trace(M_raw):.2f}) →",
          'richtig' if abs(M_raw[0, 1] + 74 / 6) < 1e-9 and abs(Mw[1] / Mw[0] + 0.0238) < 5e-5 else 'bitte prüfen')
    print(f"d) Entfernung in Metern: Varianz {cov_matrix(raw_m)[1, 1]:.0f} m², erhaltener Anteil {100 * share_m:.2f} %")